In [1]:
import h5py
import finesse
import networkx as nx
import torch
from GNN.power_predictor import LinGNN as PowerGNN
# from GNN.train_power_predictor import PowerDataset
import torch_geometric as pyg
import numpy as np
from utils.finesse_base import base_kat
from GNN.GNN_run import run_GNN

In [2]:
# kat = """
# # Add a Laser named L0 with a power of 1 W.
# l L0 P=1

# s s1 portA=L0.p1 portB=eom1.p1 L=10

# modulator eom1 9M 0.1 order=1

# s s2 portA=eom1.p2 portB=ITM.p1 L=10

# # Input mirror of cavity.
# m ITM L=0 T=0.014 Rc=-17

# # Intra-cavity space with length of 4 km
# s CAV ITM.p2 ETM.p1 L=10

# # End mirror of cavity.
# m ETM L=0 T=5u  Rc=21

# cavity cavArm source=ITM.p2.o

# # Power detectors on reflection, circulation and transmission.
# pd ETM_p1_i_pd ETM.p1.i

# # Power detectors on reflection, circulation and transmission.
# pd ITM_p2_i_pd ITM.p2.i

# pd1 pdhI node=ITM.p1.o f=eom1.f phase=0 # In phase demodulated signal
# pd1 pdhQ node=ITM.p1.o f=eom1.f phase=90 # Quadrature phase demodulated signal

# # dof ETMz ETM.dofs.z
# # readout_rf pdh_readout ITM.p1.o f=eom1.f output_detectors=true phase=0

# # Add a lock
# lock lock_length pdhI ETM.phi -1.0673950644453318 1e-12
# """

In [3]:
# test = """
#         # Add a Laser named L0 with a power of 1 W.
#         l l0 P=1
#         bp roc_l0 l0.p1.o rc

#         # Space attaching L0 <-> m1 with length of 0 m (default).
#         s s0 l0.p1 m1.p1 20

#         # Input mirror of cavity.
#         m m1 R=0.9 T=0.1 Rc=-1934

#         # Intra-cavity space with length of 10 m.
#         s LX m1.p2 m2.p1 L=3994.47

#         # End mirror of cavity.
#         m m2 R=0.9 T=0.1 Rc=2245

#         cav cavity1 m1.p2.o

#         noxaxis()
#         """

In [4]:
# def reset_model(kat):
#     fabry_perot = finesse.Model()
#     fabry_perot.parse(kat)
#     fabry_perot.modes(maxtem=6, modes='even')
#     return fabry_perot

In [5]:
def model_to_nx_port(model):

    finesse_g = model.optical_network
    g = nx.DiGraph()
    
    for node in finesse_g.nodes():
        
        opt = node.split('.')[0]
        
        if isinstance(getattr(model, opt), finesse.components.mirror.Mirror):
            # Create feature vector
            print("add a mirror node")

            # Make sure that the attributes are evaluated to floats if they are not already (if the value is not specified in the kat script, it might be a symbolic expression that needs to be evaluated)
            _Rc = getattr(model, opt).Rcx.value.eval() if not isinstance(getattr(model, opt).Rcx.value, float) else getattr(model, opt).Rcx.value
            _R = getattr(model, opt).R.value.eval() if not isinstance(getattr(model, opt).R.value, float) else getattr(model, opt).R.value

            # Add the node to the graph with the feature vector as attributes
            g.add_node(node, Rc=_Rc, R=_R, alpha=0)
            print("Mirror node added with Rc:", _Rc, "R:", _R)

        elif isinstance(getattr(model, opt), finesse.components.laser.Laser):
            print("add a laser node")
            g.add_node(node, Rc=0, R = 0, alpha=0)
        
        else:
            print("add an unknown node")
            g.add_node(node, Rc=0, R = 0, alpha=0)
    # Access edge attributes
    for i, edge in enumerate(finesse_g.edges().data()):
        print(f"Processing edge: {edge}")
        dat = list(finesse_g.edges().data())[i][2]['owner']()
        if not isinstance(dat, finesse.components.space.Space):
            print("add a space edge")
            g.add_edge(str(edge[0]), str(edge[1]), length=0, nr=1)
        else:
            print("add an unknown edge")
            g.add_edge(str(edge[0]), str(edge[1]), length=dat.L.value if not isinstance(dat.L.value, finesse.symbols.Symbol) else dat.L.value.eval(), nr=dat.nr.value if not hasattr(dat.nr.value, 'eval') else dat.nr.value.eval())
    
    return g

In [6]:
# finesse_model = reset_model(kat)
# # finesse_model = reset_model(test)
# # finesse.tb()
# # graph = model_to_nx_port_sanitized(finesse_model)

# # out1 = finesse_model.run("run_locks(display_progress=true,pre_step=print_model_attr(ETM.phi))")
# out = finesse_model.run("noxaxis()")
# print('\n my gain', out['circ'])
graph = model_to_nx_port(base_kat)

add a laser node
add a laser node
add an unknown node
add an unknown node
add an unknown node
add an unknown node
add a mirror node
Mirror node added with Rc: -20.0 R: 0.986
add a mirror node
Mirror node added with Rc: -20.0 R: 0.986
add a mirror node
Mirror node added with Rc: -20.0 R: 0.986
add a mirror node
Mirror node added with Rc: -20.0 R: 0.986
add a mirror node
Mirror node added with Rc: 15.0 R: 0.999995
add a mirror node
Mirror node added with Rc: 15.0 R: 0.999995
add a mirror node
Mirror node added with Rc: 15.0 R: 0.999995
add a mirror node
Mirror node added with Rc: 15.0 R: 0.999995
Processing edge: ('L0.p1.o', 'eom1.p1.i', {'name': 'P1i_P2o', 'in_ref': <weakref at 0x7f7c0eef1cb0; to 'OpticalNode' at 0x7f7c0eec7f20>, 'out_ref': <weakref at 0x7f7c0eee46d0; to 'OpticalNode' at 0x7f7c0f0752e0>, 'owner': <weakref at 0x7f7c0eee5080; to 'Space' at 0x7f7c0eec67b0>, 'length': 1, 'coupling_type': <CouplingType.OPTICAL_TO_OPTICAL: 0>, 'internal': False})
add an unknown edge
Processin

In [7]:
model = PowerGNN(hidden_size=1000, num_layers=10, lin_layers=5, target_size = 1)
model.load_state_dict(torch.load('GNN/power_predictor_ligo_fixed_gat10_kan5.pt', map_location=torch.device('cpu'), weights_only=True))
model.eval()

LinGNN(
  (bnn): BatchNorm(3, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bne): BatchNorm(2, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (batch_norms): ModuleList(
    (0-7): 8 x BatchNorm(1000, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (convs): ModuleList(
    (0): TransformerConv(3, 1000, heads=1)
    (1-8): 8 x TransformerConv(1000, 1000, heads=1)
    (9): TransformerConv(1000, 800, heads=1)
  )
  (linears): ModuleList(
    (0-3): 4 x Linear(in_features=800, out_features=800, bias=True)
    (4): Linear(in_features=800, out_features=1, bias=True)
  )
)

In [8]:
data = pyg.utils.from_networkx(
    graph,
    group_node_attrs=['Rc', 'R', 'alpha'],
    group_edge_attrs=['length', 'nr']
)
data.x = torch.nan_to_num(data.x, posinf=0).float()
data.edge_attr = torch.nan_to_num(data.edge_attr, posinf=0).float()
with torch.no_grad():
    out = model(data)

name = [n for n, attrs in graph.nodes(data=True)]

print(out.shape)
for i, node in enumerate(name):
    print(f"Predicted power at node {node}: {np.exp(out[i].item())}")


torch.Size([14, 1])
Predicted power at node L0.p1.i: 0.36250202870246634
Predicted power at node L0.p1.o: 0.9997818112021161
Predicted power at node eom1.p1.i: 0.3345941978643259
Predicted power at node eom1.p1.o: 1.1738950419594942
Predicted power at node eom1.p2.i: 1.4653861936529524
Predicted power at node eom1.p2.o: 0.43022958216872564
Predicted power at node ITM.p1.i: 1.010675385496489
Predicted power at node ITM.p1.o: 1.556398452505665
Predicted power at node ITM.p2.i: 146.67430153324904
Predicted power at node ITM.p2.o: 1.5563990091172262
Predicted power at node ETM.p1.i: 65.51990393751998
Predicted power at node ETM.p1.o: 0.0006366362780800879
Predicted power at node ETM.p2.i: 4.540915290632085e-05
Predicted power at node ETM.p2.o: 0.0006366362780800879


In [9]:
# model = reset_model(kat)
# out = model.run("""Series(
#                             run_locks(display_progress=false),
#                             noxaxis(),
#                             )""")
# print("power at ETM_p1_i_pd", out['noxaxis']['ETM_p1_i_pd'])
# print("power at ITM_p2_i_pd", out['noxaxis']['ITM_p2_i_pd'])

In [10]:
# dataset_fp = PowerDataset(data_files=['/home/ma/gnn-ifosim-sid/data/fabry_perot_data.h5'])

In [11]:
# dataset_fp = PowerDataset(data_files=['/home/ma/gnn-ifosim-sid/data/fabry_perot_data_fixed.h5'])

In [12]:
# print(dataset_fp[0])
# for i in range(10):
#     print(dataset_fp[i]['pd'])

In [4]:
model_path = "GNN/models/power_predictor_ligoParams_gat10_kan5.pt"
pd_name = "ETM.p1.i"
names, powers = run_GNN(base_kat,model_path)
index = names.index(pd_name)
print(powers[index])

32.1233836196624
